# Predicting Food Allergy in Children Without a Family History

A reproducible analysis across three public datasets (NHIS 2021, NHANES 2005-2006, CHOP).

**This is a demonstration project** that quantifies how far public data
goes on this question and where it stalls. Run the cells in order. All data downloads
automatically.

See `README.md` and `NOTES.md` in the repository for full context and honest limitations.

## Cell 1 — Setup

In [ ]:
import pandas as pd
import numpy as np
import zipfile, requests
from io import BytesIO
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif
from scipy import stats
import warnings
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
print("Setup complete.")

## Cell 2 — Load NHIS 2021 (no-family-history analysis)

Children are linked to parents through household IDs to define the no-family-history subgroup.

In [ ]:
def load_zip_csv(url, inner):
    r = requests.get(url)
    zf = zipfile.ZipFile(BytesIO(r.content))
    return pd.read_csv(zf.open(inner))

nhis = load_zip_csv(
    "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/NHIS/2021/child21csv.zip",
    "child21.csv")
adults = load_zip_csv(
    "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/NHIS/2021/adult21csv.zip",
    "adult21.csv")

parents = adults[["HHX","DXFOOD_A","DXSKIN_A","ASEV_A"]].copy()
parents = parents[parents["DXFOOD_A"].isin([1,2]) |
                  parents["DXSKIN_A"].isin([1,2]) |
                  parents["ASEV_A"].isin([1,2])]
hh = parents.groupby("HHX").agg(
    pf=("DXFOOD_A", lambda x: 1 if (x==1).any() else 2),
    ps=("DXSKIN_A", lambda x: 1 if (x==1).any() else 2),
    pa=("ASEV_A",   lambda x: 1 if (x==1).any() else 2)).reset_index()
hh["any_parent_allergy"] = ((hh.pf==1)|(hh.ps==1)|(hh.pa==1)).astype(int)

children = pd.merge(nhis, hh, on="HHX", how="inner")
no_fam = children[children.any_parent_allergy==0].copy()
no_fam = no_fam[no_fam.CURFOOD_C.isin([1,2])].copy()
no_fam["has_food_allergy"] = (no_fam.CURFOOD_C==1).astype(int)
with_fam = children[children.any_parent_allergy==1].copy()
with_fam = with_fam[with_fam.CURFOOD_C.isin([1,2])].copy()
with_fam["has_food_allergy"] = (with_fam.CURFOOD_C==1).astype(int)

print(f"No-family-history children: {len(no_fam):,}")
print(f"Food allergy rate (no history):   {no_fam.has_food_allergy.mean():.1%}")
print(f"Food allergy rate (with history): {with_fam.has_food_allergy.mean():.1%}")

## Cell 3 — Demographic models and the 0.602 ceiling

In [ ]:
feats = ["AGEP_C","SEX_C","RACEALLP_C","REGION","URBRRL","PCNTKIDS_C",
         "PCNTLT18TC","POVRATTC_C","SCFAMSTR_C","FDSCAT4_C","HOUTENURE_C",
         "MEDICAID_C","HICOV_C","NATUSBORN_C","RXDG12M_C"]
md_ = no_fam[feats + ["has_food_allergy"]].dropna()
X, y = md_[feats], md_["has_food_allergy"]
cv = StratifiedKFold(5, shuffle=True, random_state=42)

auc15 = cross_val_score(LogisticRegression(max_iter=1000),
        StandardScaler().fit_transform(X), y, cv=cv, scoring="roc_auc").mean()
print(f"15-variable demographic model CV AUC: {auc15:.3f}")

num = no_fam.select_dtypes(include=[np.number]).columns
drop = ["CURFOOD_C","DXFOOD_C","DXSKIN_C","has_food_allergy","any_parent_allergy",
        "HHX","WTFA_C","pf","ps","pa"]
allv = [c for c in num if c not in drop and no_fam[c].isnull().mean()<0.5
        and no_fam[c].nunique()>1]
Xall = no_fam[allv].fillna(no_fam[allv].median())
ceil = cross_val_score(LogisticRegression(max_iter=1000),
        StandardScaler().fit_transform(Xall), no_fam["has_food_allergy"],
        cv=cv, scoring="roc_auc").mean()
print(f"Ceiling using all {len(allv)} demographic variables: {ceil:.3f}")

## Cell 4 — Load NHANES 2005-2006 (biological model)

In [ ]:
b = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/"
demo = pd.read_sas(b+"DEMO_D.XPT", format="xport")
ige  = pd.read_sas(b+"AL_IGE_D.XPT", format="xport")
cbc  = pd.read_sas(b+"CBC_D.XPT", format="xport")
cot  = pd.read_sas(b+"COT_D.XPT", format="xport")
pbcd = pd.read_sas(b+"PBCD_D.XPT", format="xport")

m = demo.merge(ige, on="SEQN").merge(cbc, on="SEQN", how="left") \
        .merge(cot, on="SEQN", how="left").merge(pbcd, on="SEQN", how="left")
kids = m[m.RIDAGEYR < 18].dropna(subset=["LBXIGE"]).copy()
kids["elevated_ige"] = (kids.LBXIGE > 100).astype(int)
print(f"Children with IgE: {len(kids):,}")
print(f"Elevated IgE rate: {kids.elevated_ige.mean():.1%}")

## Cell 5 — Variable-group contributions (NHANES)

Shows that blood-count variables, not demographics, carry the predictive signal.

In [ ]:
bio = ["LBXCOT","LBXWBCSI","LBXLYPCT","LBXMOPCT","LBXNEPCT","LBXEOPCT",
       "LBXBAPCT","LBDEONO","LBXHGB","LBXPLTSI","LBXBPB","RIDAGEYR",
       "RIAGENDR","RIDRETH1","INDFMINC"]
bd = kids[bio + ["elevated_ige"]].dropna()
yb = bd["elevated_ige"]; scb = StandardScaler()

groups = {
    "Demographics": ["RIDAGEYR","RIAGENDR","RIDRETH1","INDFMINC"],
    "Environmental (cotinine, lead)": ["LBXCOT","LBXBPB"],
    "Full CBC panel": ["LBXWBCSI","LBXLYPCT","LBXMOPCT","LBXNEPCT",
                       "LBXEOPCT","LBXBAPCT","LBDEONO","LBXHGB","LBXPLTSI"],
    "Eosinophils only": ["LBXEOPCT","LBDEONO"],
    "All biological": bio,
}
for name, cols in groups.items():
    cc = [c for c in cols if c in bd.columns]
    auc = cross_val_score(LogisticRegression(max_iter=1000),
          scb.fit_transform(bd[cc]), yb, cv=cv, scoring="roc_auc").mean()
    print(f"{name:35s} AUC {auc:.3f}")

## Cell 6 — LASSO over all NHANES variables

**Note the explicit exclusion of allergen-specific IgE variables.** Including them
causes data leakage (they ARE the allergy outcome). This exclusion is essential.

In [ ]:
allcols = kids.select_dtypes(include=[np.number]).columns
excl = ["SEQN","LBXIGE","LBDIGELC","elevated_ige","WTINT2YR","WTMEC2YR",
        "SDMVPSU","SDMVSTRA","SDDSRVYR","RIDSTATR","RIDEXMON","RIDAGEEX","RIDAGEMN"]
excl += [c for c in kids.columns if c.startswith("LBD")]
# CRITICAL: exclude allergen-specific IgE (data leakage)
excl += [c for c in kids.columns if c.startswith("LBX") and len(c) > 4
         and c[3:5] in ["ID","IE","II","IM","IF","IW","IG","IT","F1","F2","F4","W1","E7"]]
cand = [c for c in allcols if c not in excl and kids[c].isnull().mean()<0.3
        and kids[c].nunique()>1]
ld = kids[cand+["elevated_ige"]].copy()
Xl = ld[cand].fillna(ld[cand].median()); yl = ld["elevated_ige"]
lasso = LogisticRegressionCV(Cs=10, cv=3, penalty="l1", solver="liblinear",
        scoring="roc_auc", max_iter=2000, random_state=42)
lasso.fit(StandardScaler().fit_transform(Xl), yl)
co = pd.DataFrame({"feature":cand,"coef":lasso.coef_[0]})
print(f"{(co.coef.abs()>0.001).sum()} of {len(cand)} variables survived")
print("Strongest predictor:", co.loc[co.coef.abs().idxmax(),"feature"])

## Cell 7 — CHOP eczema validation

In [ ]:
chop = pd.read_csv(
    "https://zenodo.org/records/44529/files/food-allergy-analysis-Zenodo.csv?download=1")
chop["early_eczema"] = (chop["ATOPIC_DERM_START"] < 2).astype(int)
chop["fa"] = (chop["PEANUT_ALG_START"].notna() | chop["EGG_ALG_START"].notna() |
              chop["MILK_ALG_START"].notna() | chop["SHELLFISH_ALG_START"].notna() |
              chop["TREENUT_ALG_START"].notna()).astype(int)
cvc = chop[chop["AGE_END_YEARS"]>=3]
a = int(cvc[cvc.early_eczema==1].fa.sum()); bb=int((cvc.early_eczema==1).sum()-a)
c = int(cvc[cvc.early_eczema==0].fa.sum()); d=int((cvc.early_eczema==0).sum()-c)
OR = (a/bb)/(c/d)
se = np.sqrt(1/a+1/bb+1/c+1/d)
lo, hi = np.exp(np.log(OR)-1.96*se), np.exp(np.log(OR)+1.96*se)
print(f"Eczema before age 2 -> food allergy")
print(f"OR = {OR:.2f} (95% CI {lo:.2f}-{hi:.2f}), n={len(cvc):,}")

## Summary

- 7.0% of no-family-history children have food allergy (NHIS).
- Demographic prediction ceiling ~0.602; blood-count model ~0.718 (NHANES).
- Eosinophil percentage is the dominant predictor across methods.
- Eczema before age 2 is associated with later food allergy (CHOP).

See `NOTES.md` for limitations.